<a href="https://colab.research.google.com/github/sudenazgrmz/TechSupportPortal/blob/main/notebooks/03_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_score

df = pd.read_csv("deceptive-opinion.csv")

X = df["text"]                                   # only the review text
y = (df["deceptive"] == "deceptive").astype(int) # 1 = deceptive, 0 = truthful
groups = df["hotel"]                             # used only for splitting

models = {
    "tfidf_logreg": make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2),
        LogisticRegression(max_iter=1000),
    ),
    "tfidf_svm": make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=2),
        LinearSVC(),
    ),
}

random_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
group_cv = GroupKFold(n_splits=5)

for name, model in models.items():
    scores_random = cross_val_score(model, X, y, cv=random_cv, scoring="f1_macro")
    scores_group = cross_val_score(model, X, y, cv=group_cv, groups=groups, scoring="f1_macro")
    print(name)
    print("  random split  macro-F1: mean", round(scores_random.mean(), 3),
          "std", round(scores_random.std(), 3))
    print("  grouped split macro-F1: mean", round(scores_group.mean(), 3),
          "std", round(scores_group.std(), 3))
    print()

tfidf_logreg
  random split  macro-F1: mean 0.885 std 0.014
  grouped split macro-F1: mean 0.87 std 0.015

tfidf_svm
  random split  macro-F1: mean 0.896 std 0.008
  grouped split macro-F1: mean 0.884 std 0.018



In [ ]:
model = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2),
    LogisticRegression(max_iter=1000),
)
model.fit(X, y)   # fitted on ALL data, only for inspection, not for evaluation

vec = model.named_steps["tfidfvectorizer"]
clf = model.named_steps["logisticregression"]
names = np.array(vec.get_feature_names_out())
order = np.argsort(clf.coef_[0])

print("Top 25 features pointing to TRUTHFUL:")
print(list(names[order[:25]]))
print()
print("Top 25 features pointing to DECEPTIVE:")
print(list(names[order[-25:][::-1]]))

Top 25 features pointing to TRUTHFUL:
['location', 'floor', 'great', 'on', 'small', 'we', 'no', 'bathroom', 'is', 'rate', 'michigan', 'breakfast', 'street', 'large', 'reviews', 'priceline', 'good', 'bed', 'walk', 'quiet', 'river', 'helpful', 'excellent', 'coffee', 'for']

Top 25 features pointing to DECEPTIVE:
['chicago', 'my', 'luxury', 'hotel', 'experience', 'my stay', 'was', 'my room', 'chicago hotel', 'when', 'husband', 'vacation', 'be', 'my husband', 'hilton', 'staying', 'to', 'food', 'definitely', 'luxurious', 'the food', 'husband and', 'recently', 'this hotel', 'millennium']
